# Climatch: be a data detective, step by step
Run each cell from top to bottom with **Shift + Enter**. Read the text above each cell first.

The question for every factor: **What happened? Is it real? Why does it matter? What does it likely mean?** No predictions.

## 1. Load the tools
If this fails with `ModuleNotFoundError`, click the kernel name (top right) and pick `.venv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # the data_pipeline folder
import pandas as pd
import matplotlib.pyplot as plt
import pipeline as p

th = p.load_thresholds()
print('Factors in thresholds.json:', [k for k in th if k not in ('_note', 'version', 'settings')])
from paths import load_json

## 2. Download daily NASA POWER data
First run: 10–60 seconds. After that it reads the saved copy in `cache/`.

In [ ]:
LAT, LON = 33.45, -112.07   # Phoenix. Change to try another place.
raw = p.fetch_power(LAT, LON)
raw.head(10)

## 3. Look for problems BEFORE cleaning
`-999` is NASA's code for 'no data'.

In [ ]:
print('Days:', len(raw))
print((raw == -999).sum())
raw.describe().round(1)

## 4. Clean

In [ ]:
df, valid_per_year, bad_years = p.clean_daily(raw)
print('Incomplete years (left out):', bad_years)
valid_per_year.tail()

## 5. Sanity-check plots for one year
Phoenix summer max should be ~40–45 °C; rain should come in a few big spikes (summer monsoon, winter storms).

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df.loc['2024', ['T2M_MAX', 'T2M_MIN']].plot(ax=ax[0], title='Daily max / min temperature (°C)')
ax[0].axhline(35, color='red', ls='--'); ax[0].axhline(20, color='blue', ls='--'); ax[0].axhline(0, color='gray', ls=':')
df.loc['2024', 'PRECTOTCORR'].plot(ax=ax[1], kind='area', title='Daily rain (mm)')
ax[1].axhline(20, color='red', ls='--')
plt.tight_layout(); plt.show()

## 6. Days → yearly counts (the dots on each chart)

In [ ]:
flags = p.daily_flags(df, th)
annual = p.annual_counts(flags, bad_years)
annual.plot(marker='o', subplots=True, layout=(2, 3), figsize=(14, 6), legend=True)
plt.tight_layout(); plt.show()
annual.tail()

## 7. Remove the seasons (deseasonalized anomaly)
Every month minus the usual value for that month, so summer doesn't look like a trend.

In [ ]:
monthly = p.monthly_counts(flags)['hot_days']
anom = p.monthly_anomaly(monthly)
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
monthly.plot(ax=ax[0], title='Very hot days per month')
anom.plot(ax=ax[1], title='Same, seasons removed (anomaly)', color='purple'); ax[1].axhline(0, color='k', lw=.5)
plt.tight_layout(); plt.show()

## 8. Trend + significance for very hot days
Try changing `START` to 2016 (10 years). Does it stay significant?

In [ ]:
START, END = 2001, 2025
tr = p.trend_stats(annual['hot_days'], START, END)
for k in ['then_years', 'then', 'now_years', 'now', 'slope_per_decade', 'slope_ci95_per_decade', 'significance_label', 'p_label']:
    print(f'{k:24s}', tr[k])

## 9. Draw the chart like the website (real data only, no extension)

In [ ]:
line, pts = pd.DataFrame(tr['line']), pd.DataFrame(tr['points'])
fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(line.year, line.lo95, line.hi95, alpha=0.3, label='95% band for the trend')
ax.plot(line.year, line.trend, label=f"Trend {tr['slope_per_decade']:+.1f} days/decade ({tr['p_label']})")
ax.scatter(pts.year, pts.value, color='black', s=15, label='Each year')
for yrs, val, c in [(tr['then_years'], tr['then'], 'tab:blue'), (tr['now_years'], tr['now'], 'tab:red')]:
    ax.hlines(val, yrs[0], yrs[1], colors=c, lw=4)
ax.set_title('Very hot days per year: then (blue) vs now (red)'); ax.legend(); plt.show()

## 10. The 4-sentence story the website shows

In [ ]:
cfg = th['hot_days']
narr = load_json('narratives.json')['hot_days']
for k, s in p.describe(tr, cfg, narr).items():
    print(f'{k}:\n  {s}\n')

## 11. Team decision: which temperature for 'very hot'?

In [ ]:
p.compare_heat_cutoffs(LAT, LON)

## 12. The full report for a person (condition + age)
Other data sources (PM2.5, NO₂, fires, MODIS heat) show `not available` until you build them (GUIDE.md parts E–H). That's fine.

In [ ]:
rep = p.build_report(LAT, LON, condition='asthma_copd', age_group='65_plus', sources=('pm25', 'no2', 'fires'))
p.print_report(rep)
print('\nCard order for this person:', rep['card_order'])